# Beispiel: Titanic komplett

Wer hat den Untergang der Titanic überlebt? Zwei Durchläufe nach den Schritten der Ablauf-Grafik aus dem Kapitel Supervised ML: **Teil 1** ein einfaches Modell (logistische Regression, 10 Schritte, mit gründlicher EDA), **Teil 2** ausführlich mit Feature Engineering und Random Forest (11 Schritte). Die Teile sind unabhängig, jeder lädt die Daten selbst. Dieselbe Strecke mit Erklärungen und fertigen Grafiken steht in der Lernapp auf der Seite `titanic.html`.

- **Herkunft:** selbst erstellt für die Lernapp, kein HSLU-Kursmaterial. Daten: der öffentliche Kaggle-Datensatz Titanic (`train.csv`, 891 Passagiere, Ziel `Survived`). Das Notebook nimmt `titanic/data.csv` aus deinem Gruppen-Repo, falls vorhanden, sonst lädt es die identische öffentliche Kopie aus dem Netz (dann braucht es Internet).
- **Ausführen:** von oben nach unten, das Notebook hängt von keinem anderen ab. Lokal in der Lernapp oder auf jedem anderen PC im Browser: [In Colab öffnen](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/beispiele/07_Titanic.ipynb) (ein Google-Konto genügt, solange das Repo öffentlich ist; ist es privat, in Colab mit GitHub anmelden und «Include private repos» anhaken).
- **Navigation:** Das Inhaltsverzeichnis links in JupyterLab springt zu jedem Abschnitt. Zurück zur Übersicht: Lernapp, Kapitel «Übersicht».

In [ ]:
# Colab has scikit-learn, pandas and matplotlib; umap and imblearn are installed there on demand
import importlib.util, subprocess, sys
fehlend = [pip for mod, pip in [("umap", "umap-learn"), ("imblearn", "imbalanced-learn")] if importlib.util.find_spec(mod) is None]
if fehlend and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *fehlend], check=True)
elif fehlend:
    print("Für einzelne Beispiele fehlt: pip install " + " ".join(fehlend))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import (load_diabetes, load_breast_cancer, load_wine, load_digits,
                              make_moons, make_blobs, make_classification)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, roc_auc_score, silhouette_score,
                             adjusted_rand_score)

SEED = 42
plt.rcParams["figure.figsize"] = (6, 3.5)
pd.set_option("display.precision", 3)

## Daten

In [ ]:
from pathlib import Path

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"   # same 891 rows as Kaggle train.csv
LOKAL = Path("../../../titanic/data.csv")                                                 # local copy next to the lernapp folder, if present


def lade_titanic():
    """Titanic training data: local file if present, otherwise the public copy."""
    df = pd.read_csv(LOKAL if LOKAL.exists() else URL)
    return df.dropna(subset=["Survived"]).astype({"Pclass": int, "Survived": int})


lade_titanic().shape

## 1 · Einfaches Modell: logistische Regression in 10 Schritten

Diese Version folgt genau den 10 Schritten der Ablauf-Grafik im Kapitel Supervised ML. Daten: Kaggle-Titanic (`train.csv`, hier unter `titanic/data.csv`, 891 Passagiere, Ziel `Survived`). Die Teile laufen nacheinander wie Zellen in einem Notebook. Ausgaben und Bilder stammen aus genau diesem Code (scikit-learn 1.9).

### Schritt 1 · Rohdaten laden

Die Kaggle-Datei `train.csv` hat 891 Passagiere und 12 Spalten. Am Dateiende steht eine leere Zeile, die `dropna(subset=["Survived"])` entfernt. Danach liest pandas `Pclass` und `Survived` als Kommazahlen, `astype(int)` macht wieder ganze Zahlen daraus. Schon der erste Blick zeigt die Baustellen: Das Alter fehlt bei 177 Passagieren, die Kabine bei 687, der Einstiegshafen bei 2.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

# 1 · Rohdaten laden
df = lade_titanic()  # 1 leere Zeile
print(df.shape, "| fehlend:", df[["Age", "Cabin", "Embarked"]].isna().sum().to_dict())

### Schritt 2 · Aufteilen und Schritt 3 · EDA

Bevor irgendetwas ausgewertet wird, kommen 20 % der Passagiere (179) als Testset weg. `stratify=y` sorgt dafür, dass in beiden Teilen gleich viele Überlebende sind (rund 38 %), `random_state=0` macht den Split wiederholbar.

Die EDA läuft nur auf den 712 Trainingspassagieren. `assign` hängt das Ziel wieder an, damit man Überlebensraten nach Gruppen rechnen kann.

In [ ]:
# 2 · Aufteilen: Testset sofort wegsperren
X, y = df.drop(columns="Survived"), df["Survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

# 3 · EDA, nur auf den Trainingsdaten
eda = X_tr.assign(Survived=y_tr)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
eda.groupby(["Pclass", "Sex"])["Survived"].mean().unstack().plot.bar(ax=ax[0], rot=0, title="Überlebensrate nach Klasse und Geschlecht")
eda["Age"].plot.hist(bins=30, ax=ax[1], title="Alter (fehlt bei rund 20 %)")
plt.show()

*Grafik:* Frauen und die 1. Klasse überlebten viel häufiger. Das Alter fehlt oft und ist schief verteilt, darum Median statt Mittelwert.

### Schritt 3 vertieft · Die Daten gründlich ansehen

Die Grafik oben ist nur der erste Blick. Eine gründliche EDA geht jede Spalte durch: Was fehlt, wie sind die Werte verteilt, was hängt mit dem Überleben zusammen und was miteinander? Alles weiterhin nur auf den 712 Trainingspassagieren.

### 3a · Überblick

In [ ]:
# 3a · Überblick: die Zahlen-Spalten zusammengefasst
import numpy as np, seaborn as sns
eda = eda.astype({"SibSp": int, "Parch": int})  # wegen der leeren Zeile als Kommazahl gelesen
print(len(eda), "Trainingspassagiere, Überlebensrate:", round(eda["Survived"].mean(), 3))
print(eda[["Age", "Fare", "SibSp", "Parch"]].describe().round(1))

`describe()` fasst jede Zahlen-Spalte zusammen. Auffällig: `Age` hat nur 566 von 712 Werten (`count`). Bei `Fare` liegt der Mittelwert (33,1) weit über dem Median (14,5), das Maximum ist 512: Wenige Passagiere zahlten sehr viel, die Verteilung ist schief. `SibSp` und `Parch` sind meist 0, die meisten reisten allein.

Die Überlebensrate von 38 % ist die Messlatte für jedes Modell: Wer stur für alle «gestorben» vorhersagt, hat schon 62 % Accuracy. Ein Modell muss deutlich besser sein.

### 3b · Fehlende Werte

In [ ]:
# 3b · Fehlende Werte: wo und wie viele?
fehlend = eda.isna().mean().sort_values(ascending=False)
print((fehlend[fehlend > 0] * 100).round(1))
fig, ax = plt.subplots(figsize=(9, 3))
sns.heatmap(eda.isna().T, cbar=False, cmap="Greys", xticklabels=False, ax=ax)
ax.set_title("Fehlende Werte: schwarz = fehlt (eine Spalte pro Passagier)")
plt.show()

*Grafik:* Jeder schwarze Strich ist ein fehlender Wert. Cabin fehlt fast immer, Age oft, Embarked nur zweimal. Alle anderen Spalten sind vollständig.

Daraus folgen drei verschiedene Entscheidungen. `Cabin` fehlt bei 77,5 %: Das ist zu viel zum Auffüllen, die Spalte bleibt draussen (Schritt 4). `Age` fehlt bei 20,5 %, ist aber zu wichtig zum Weglassen: auffüllen. `Embarked` fehlt nur bei 2 Passagieren: mit dem häufigsten Hafen füllen.

### 3c · Verteilungen und Ausreisser

In [ ]:
# 3c · Verteilungen und Ausreisser: Fare
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
eda["Fare"].plot.hist(bins=40, ax=ax[0], title="Fare: stark schief")
np.log1p(eda["Fare"]).plot.hist(bins=40, ax=ax[1], title="log(1 + Fare): ausgeglichener")
sns.boxplot(data=eda, x="Pclass", y="Fare", ax=ax[2]).set_title("Fare nach Klasse")
plt.show()
print("Median-Fare nach Klasse:", eda.groupby("Pclass")["Fare"].median().round(1).to_dict())
print("Fare über 200:", (eda["Fare"] > 200).sum(), "Passagiere")

*Grafik:* Links: Fare ist stark rechtsschief. Mitte: Der Logarithmus staucht die grossen Werte. Rechts: Die teuren Tickets gehören fast alle zur 1. Klasse.

Die Ausreisser bei `Fare` (19 Passagiere über 200) sind keine Tippfehler: Der Boxplot zeigt, dass sie zur 1. Klasse gehören, deren Median-Preis mit 66,6 achtmal so hoch ist wie in der 3. Klasse (8,0). Echte Ausreisser behält man. `log1p` rechnet log(1 + x), damit die Tickets mit Preis 0 (der Balken ganz links in der Mitte) kein Problem machen. Wichtig: Der `StandardScaler` verschiebt und staucht nur, die schiefe Form bleibt. Für ein lineares Modell kann darum `log(Fare)` besser sein, ein Baum braucht das nicht.

### 3d · Überlebensrate je Merkmal

In [ ]:
# 3d · Überlebensrate je Merkmal
fig, ax = plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for a, col in zip(ax, ["Sex", "Pclass", "Embarked", "SibSp"]):
    eda.groupby(col)["Survived"].mean().plot.bar(ax=a, rot=0, title=col)
ax[0].set_ylabel("Überlebensrate")
plt.show()
print(eda.groupby("Embarked")["Pclass"].value_counts(normalize=True).unstack().round(2))

*Grafik:* Geschlecht und Klasse trennen am stärksten. Der Hafen und SibSp zeigen schwächere Unterschiede.

**Geschlecht** ist das stärkste Merkmal: 75 % der Frauen, aber nur 19 % der Männer überlebten («Frauen und Kinder zuerst»). Danach kommt die **Klasse**: 63 % in der 1., 49 % in der 2., 23 % in der 3. Klasse.

Beim **Einstiegshafen** sieht Cherbourg (C, 53 %) besser aus als Southampton (S, 35 %). Die Tabelle zeigt den Grund: Gut die Hälfte der Passagiere aus Cherbourg reiste 1. Klasse, aus Queenstown (Q) fast alle 3. Klasse. Der Hafen wirkt also vor allem über die Klasse. So eine dritte Grösse im Hintergrund heisst *Confounder*: Der Hafen selbst hat niemanden gerettet.

Bei **SibSp** ist der Zusammenhang nicht linear: Mit einem Geschwister oder Partner war die Rate am höchsten (55 %), allein (34 %) und in grossen Familien tiefer. Bei 5 und 8 sind es nur je 4 Passagiere, alle gestorben, darum fehlen dort die Balken. So kleine Gruppen sind statistisch kaum aussagekräftig.

### 3e · Alter und Überleben

In [ ]:
# 3e · Alter nach Überleben
fig, ax = plt.subplots(figsize=(7, 3))
for s, name in [(0, "gestorben"), (1, "überlebt")]:
    eda.loc[eda["Survived"] == s, "Age"].plot.hist(bins=30, alpha=0.5, ax=ax, label=name)
ax.set_xlabel("Alter"); ax.set_title("Alter nach Überleben"); ax.legend()
plt.show()
print("Überlebensrate unter 10 Jahren:", round(eda.loc[eda["Age"] < 10, "Survived"].mean(), 2))

*Grafik:* Bei kleinen Kindern überwiegen die Überlebenden, bei den 20- bis 30-Jährigen die Toten.

Kinder unter 10 Jahren überlebten zu 57 %, deutlich mehr als der Durchschnitt (38 %). Zwischen 20 und 30 überwiegen die Toten, das sind vor allem junge Männer in der 3. Klasse. Über alle Alter gesehen ist der Zusammenhang aber nicht gerade: Ein lineares Modell kann «unter 10 ist gut» nur schlecht abbilden, ein Baum lernt so eine Schwelle direkt.

### 3f · Korrelations-Heatmap

In [ ]:
# 3f · Korrelations-Heatmap (Geschlecht als 0/1)
num_eda = eda[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]].assign(Frau=(eda["Sex"] == "female").astype(int))
plt.figure(figsize=(6.5, 5))
sns.heatmap(num_eda.corr().round(2), annot=True, cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Korrelationen (nur Trainingsdaten)")
plt.show()

*Grafik:* Rot: steigen zusammen. Blau: das eine steigt, das andere sinkt. Weiss: kein linearer Zusammenhang.

Mit dem Ziel `Survived` hängen am stärksten zusammen: `Frau` (+0,55), `Pclass` (−0,35, höhere Klassennummer heisst weniger Überlebende) und `Fare` (+0,27). Das bestätigt die Säulendiagramme.

Zwischen den Merkmalen fällt `Pclass` gegen `Fare` auf (−0,55): Teure Tickets sind 1. Klasse, die beiden Spalten tragen teilweise dieselbe Information. `SibSp` und `Parch` hängen zusammen (+0,43), das spricht dafür, sie zu einer Familiengrösse zusammenzufassen (Teil 2).

Die Korrelation misst nur gerade Zusammenhänge. Beim Alter zeigt sie −0,06, obwohl Kinder klar häufiger überlebten (3e). Bei Kategorien wie `Pclass` ist sie nur eine grobe Näherung. Und Korrelation ist keine Ursache, siehe den Einstiegshafen.

### Was aus der EDA folgt

| Befund | Folge | Schritt |
|---|---|---|
| `Cabin` fehlt zu 77 % | weglassen | 4 |
| `PassengerId`, `Name`, `Ticket` sind Kennungen | weglassen (der Titel im Namen kommt in Teil 2) | 4 |
| `Age` fehlt zu 20 %, schief verteilt | mit dem Median füllen | 5 |
| `Embarked` fehlt zweimal | mit dem häufigsten Wert füllen | 5 |
| `Sex`, `Pclass`, `Embarked` sind Kategorien | One-Hot | 5 |
| Age bis 80, Fare bis 512: verschiedene Grössenordnungen | skalieren | 6 |
| Fare-Ausreisser sind echt (1. Klasse) | behalten | 5 |
| 62 % gestorben, 38 % überlebt | Accuracy muss klar über 62 % liegen; `stratify` beim Split | 2, 8 |

### Schritt 4 bis 9 · Merkmale, Pipeline, Training

**Schritt 4, Feature Selection (manuell):** `PassengerId` und `Name` sind Kennungen, `Ticket` ist fast für jeden Passagier anders, `Cabin` fehlt zu 77 %. Diese Spalten bleiben draussen. Übrig sind vier Zahlen und drei Kategorien.

**Schritt 5 und 6, Vorverarbeitung und Skalieren:** Der `ColumnTransformer` behandelt Zahlen und Kategorien getrennt. Zahlen: fehlende Werte mit dem Median füllen, dann skalieren. Kategorien: fehlende Werte mit dem häufigsten Wert füllen (betrifft `Embarked`), dann One-Hot. Weil alles in der Pipeline steckt, lernt jeder Schritt seine Werte (Median, Mittelwert, Kategorien) nur aus den Trainingsdaten des jeweiligen Folds.

**Schritt 7, Feature Selection (automatisch):** Die L1-Strafe (`l1_ratio=1`) ist Embedded Feature Selection. Sie drückt die Gewichte unwichtiger Merkmale auf genau 0.

**Schritt 8 und 9, Trainieren, validieren, bestes Modell wählen:** `GridSearchCV` probiert vier Werte für `C` aus. Kleines `C` heisst starke Strafe, also ein einfacheres Modell. Jeder Wert wird mit 5-facher Cross-Validation bewertet, das sind 20 Trainingsläufe. Danach trainiert `GridSearchCV` das Modell mit dem besten `C` automatisch noch einmal auf allen 712 Trainingspassagieren (`refit=True` ist die Voreinstellung).

In [ ]:
# 4 · Feature Selection (manuell): ID, Name, Ticket und fast leere Cabin raus
num, cat = ["Age", "Fare", "SibSp", "Parch"], ["Pclass", "Sex", "Embarked"]

# 5 + 6 · Vorverarbeitung und Skalieren, in der Pipeline nur auf Train gelernt
prep = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), cat))

# 7 · Feature Selection (automatisch): L1 setzt unwichtige Gewichte auf 0
model = make_pipeline(prep, LogisticRegression(l1_ratio=1, solver="liblinear"))  # vor sklearn 1.8: penalty="l1"

# 8 + 9 · Mit 5-facher Cross-Validation trainieren und validieren, bestes C wählen
search = GridSearchCV(model, {"logisticregression__C": [0.01, 0.1, 1, 10]}, cv=5)
search.fit(X_tr[num + cat], y_tr)
print("bestes C:", search.best_params_["logisticregression__C"], "| CV-Accuracy:", round(search.best_score_, 3))
coef = search.best_estimator_[-1].coef_[0]
print("Merkmale mit Gewicht 0:", (coef == 0).sum(), "von", len(coef))

Nach One-Hot sind es 12 Spalten: 4 Zahlen, 3 für `Pclass`, 2 für `Sex`, 3 für `Embarked`. Bei 3 davon hat L1 das Gewicht auf 0 gesetzt, das Modell kommt ohne sie aus. Die CV-Accuracy von 0,803 ist die Schätzung, wie gut das Modell auf neuen Passagieren sein wird.

### Schritt 10 · Einmal auf dem Testset bewerten

Erst jetzt kommen die 179 weggesperrten Passagiere ins Spiel, und zwar genau einmal. `predict` schickt sie durch dieselbe Pipeline, mit den Werten aus dem Training (`transform`, kein neues `fit`).

In [ ]:
# 10 · Genau einmal auf dem Testset bewerten
pred = search.predict(X_te[num + cat])
print("Test-Accuracy:", round(accuracy_score(y_te, pred), 3))
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["gestorben", "überlebt"])
plt.show()

*Grafik:* Confusion Matrix auf den 179 Testpassagieren. Die Test-Accuracy (0,79) liegt nahe an der CV-Schätzung (0,80): Die CV war verlässlich, es ist kein Testwissen eingeflossen. Dass das Modell nicht überangepasst ist, zeigt der Vergleich mit dem Training (Accuracy 0,805 gegenüber CV 0,803).

Bewusst weggelassen: Feature Engineering (Familiengrösse, Titel aus dem Namen), andere Modelle und Metriken neben der Accuracy. Das alles zeigt Teil 2.

## 2 · Ausführlich: Random Forest mit Feature Engineering

Diese Version folgt den 11 Schritten der Titanic-Aufgabe aus dem Kurs (Kapitel Klassifikation). Sie baut neue Merkmale, gleicht das Klassenverhältnis aus, misst mehrere Metriken und sucht Hyperparameter für einen Random Forest. Ausgaben und Bilder stammen aus genau diesem Code (scikit-learn 1.9).

1. Daten ansehen (Struktur, Verteilungen, Anteil der Klassen).
2. Aufräumen und Kategorien in Zahlen umwandeln (One-Hot).
3. Neue Merkmale bauen (z. B. Familiengrösse = `SibSp + Parch + 1`).
4. Fehlende Werte füllen (z. B. Alter mit dem Median).
5. Merkmale auswählen.
6. Klassenverhältnis prüfen, bei Bedarf Resampling oder `class_weight`.
7. Train/Test trennen, Cross-Validation auf dem Training.
8. Mit passenden Metriken bewerten (F1, AUC, nicht nur Accuracy).
9. Auf Leakage prüfen.
10. Regularisierung und Hyperparameter-Suche.
11. Bestes Modell **einmal** auf dem Test-Set prüfen.

### Schritt 1 · Daten ansehen

Neben der Form zeigt die Ausgabe den Anteil fehlender Werte (Alter 20 %, Kabine 77 %) und das Klassenverhältnis: 62 % sind gestorben, 38 % haben überlebt. Das ist leicht unausgewogen, aber noch kein Problem wie bei 99 zu 1.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay

# 1 · Daten ansehen: Struktur, fehlende Werte, Anteil der Klassen
df = lade_titanic()
print(df.shape)
print("fehlend:", df.isna().mean().round(2)[lambda s: s > 0].to_dict())
print("überlebt:", df["Survived"].value_counts(normalize=True).round(2).to_dict())

### Schritt 2 und 3 · Aufräumen und neue Merkmale bauen

**Feature Engineering:** `FamilySize` fasst Geschwister/Partner und Eltern/Kinder zusammen. Aus dem Namen wird der Titel gelesen («Braund, Mr. Owen» wird zu «Mr»). Seltene Titel wie «Dr» oder «Rev» kommen in die Gruppe «Rare», sonst hätte One-Hot Spalten mit nur einem Passagier.

Diese neuen Spalten rechnet man Zeile für Zeile aus, ohne Werte aus anderen Zeilen zu lernen. Darum dürfen sie vor dem Split entstehen. Die Auswertung mit `groupby` schaut dagegen auch auf die späteren Testpassagiere. Die saubere Auswertung nur auf den Trainingsdaten folgt weiter unten, direkt nach dem Split.

In [ ]:
# 2 + 3 · Aufräumen und neue Merkmale bauen
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["Title"] = df["Name"].str.split(", ").str[1].str.split(".").str[0]        # "Braund, Mr. Owen" -> "Mr"
df["Title"] = df["Title"].where(df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
print(df.groupby("Title")["Survived"].agg(["mean", "size"]).round(2))

Der Titel sagt mehr als das Geschlecht: «Master» sind Jungen, und die überlebten viel häufiger (57 %) als erwachsene Männer («Mr», 16 %). Das Merkmal fasst Geschlecht und Altersgruppe in einer Spalte zusammen.

### Schritt 4 bis 8 · Merkmale, Split, Pipeline, Cross-Validation

Die Nummern im Code folgen der Kursliste oben, darum stehen sie nicht der Reihe nach. Die Reihenfolge im Code ist dieselbe wie in der Ablauf-Grafik: zuerst Merkmale wählen und das Testset abtrennen, dann die Vorverarbeitung in der Pipeline festlegen.

**Schritt 5, Merkmale auswählen:** `Name` (steckt jetzt im Titel), `Ticket`, `Cabin` und `PassengerId` bleiben draussen, ebenso `SibSp` und `Parch` (stecken in `FamilySize`).

In [ ]:
# 5 · Merkmale auswählen (Name, Ticket, Cabin, PassengerId bleiben draussen)
num = ["Age", "Fare", "FamilySize"]
cat = ["Pclass", "Sex", "Embarked", "Title"]
X, y = df[num + cat], df["Survived"]

# 7 · Train/Test trennen, Testset wegsperren (stratify: gleicher Anteil Überlebende)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

### Die neuen Merkmale ansehen, nur auf den Trainingsdaten

Jetzt, nach dem Split, die saubere Auswertung der neuen Merkmale: nur auf den 712 Trainingspassagieren.

In [ ]:
# 1 (Fortsetzung) · Die neuen Merkmale ansehen, nur auf den Trainingsdaten
import seaborn as sns
tr = X_tr.assign(Survived=y_tr, FamilySize=X_tr["FamilySize"].astype(int))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
tr.groupby("Title")["Survived"].mean().sort_values().plot.barh(ax=ax[0], title="Überlebensrate nach Titel")
tr.groupby("FamilySize")["Survived"].mean().plot.bar(ax=ax[1], rot=0, title="Überlebensrate nach Familiengrösse")
plt.show()
print(tr.groupby("FamilySize")["Survived"].agg(["mean", "size"]).round(2))

*Grafik:* Links: Der Titel trennt Jungen (Master) von Männern (Mr). Rechts: Familien mit 2 bis 4 Personen überlebten am häufigsten.

Beim **Titel** liegen Mrs (80 %) und Miss (70 %) vorne, Mr mit 15 % ganz hinten. Master, also Jungen, liegt mit 57 % weit über Mr: Der Titel trennt Buben von Männern, was das Geschlecht allein nicht kann.

Die **Familiengrösse** zeigt einen Buckel: Allein Reisende 30 %, Familien mit 2 bis 4 Personen 56 bis 74 %, ab 5 Personen deutlich weniger. Ein lineares Modell kann so einen Buckel nicht abbilden, ein Random Forest schon. Die Tabelle zeigt aber auch: Die grossen Familien sind wenige (zum Beispiel 4 Passagiere mit Familiengrösse 11), ihre Raten sind unsicher.

In [ ]:
# Heatmap mit den neuen Merkmalen (Kategorien als 0/1-Spalten)
dummies = pd.get_dummies(tr[["Survived", "Age", "Fare", "FamilySize", "Pclass", "Sex", "Title"]],
                         columns=["Sex", "Title"], dtype=int).drop(columns="Sex_male")
plt.figure(figsize=(8, 6.5))
sns.heatmap(dummies.corr().round(2), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, annot_kws={"size": 8})
plt.title("Korrelationen mit den neuen Merkmalen (nur Trainingsdaten)")
plt.show()

*Grafik:* Die Titel-Spalten hängen stark mit dem Geschlecht zusammen, Title_Mr fast vollständig.

`Title_Mr` und `Sex_female` hängen mit −0,86 fast vollständig zusammen: Wer «Mr» ist, ist ein Mann. Die beiden Spalten tragen weitgehend dieselbe Information. Für einen Random Forest ist das kein Problem. Bei einem linearen Modell würden die Gewichte dadurch unsicher (*Multikollinearität*), dort würde man eine der beiden weglassen oder regularisieren.

`Title_Master` korreliert mit dem Alter (−0,42) und der Familiengrösse (+0,41): Jungen reisten mit Familie. `FamilySize` selbst korreliert mit dem Überleben fast gar nicht (0,02), obwohl das Säulendiagramm einen klaren Zusammenhang zeigt. Der Buckel ist nicht gerade, die Korrelation sieht ihn nicht. Darum immer beides anschauen: Grafik und Zahl.

### Pipeline, Klassenverhältnis und Cross-Validation

**Schritt 2 und 4, One-Hot und fehlende Werte:** gleich wie in Teil 1, aber mit benannten Schritten (`Pipeline` statt `make_pipeline`). Die Namen braucht man später für das Grid (`clf__max_depth`). Ein Random Forest bräuchte keine Skalierung, sie schadet aber nicht.

**Schritt 6, Klassenverhältnis:** `class_weight="balanced"` gewichtet die seltenere Klasse (Überlebende) stärker, statt Zeilen zu kopieren oder zu löschen.

**Schritt 7 und 8, Cross-Validation mit mehreren Metriken:** `StratifiedKFold` hält den Anteil Überlebender in jedem Fold gleich. `cross_validate` misst Accuracy, F1 und AUC, jeweils auf dem Trainings- und dem Validierungsteil.

In [ ]:
# 2 + 4 · One-Hot und fehlende Werte füllen, in der Pipeline nur auf Train gelernt
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat)])

# 6 · Klassenverhältnis 62:38, leicht unausgewogen: class_weight gleicht es aus
model = Pipeline([("prep", prep),
                  ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0))])

# 7 + 8 · Cross-Validation auf dem Training, mit mehreren Metriken
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
res = cross_validate(model, X_tr, y_tr, cv=cv, scoring=["accuracy", "f1", "roc_auc"], return_train_score=True)
print(pd.DataFrame(res).filter(regex="test|train").mean().round(3))

Auf den Trainingsfolds ist der Wald fast perfekt (Accuracy 0,985), auf den Validierungsfolds nur 0,80. Dieser grosse Abstand ist das typische Zeichen für Overfitting: Die Bäume lernen die Trainingspassagiere auswendig. F1 (0,75) liegt unter der Accuracy, weil die Überlebenden schwerer zu treffen sind als die Gestorbenen.

### Schritt 9 · Auf Leakage prüfen

Leakage zeigt sich oft so: Ein einzelnes Merkmal ist übermächtig, weil es die Antwort schon enthält (zum Beispiel eine Spalte mit der Nummer des Rettungsboots). Darum schaut man die Wichtigkeit der Merkmale an.

In [ ]:
# 9 · Auf Leakage prüfen: Train viel besser als CV ist Overfitting; ein einzelnes übermächtiges Merkmal wäre verdächtig
model.fit(X_tr, y_tr)
imp = pd.Series(model[-1].feature_importances_, index=model[0].get_feature_names_out())
imp.sort_values().tail(8).plot.barh(figsize=(6, 3), title="Wichtigste Merkmale (Random Forest)")
plt.show()

*Grafik:* Fare und Alter liegen vorne, danach Titel «Mr» und Geschlecht. Kein einzelnes Merkmal dominiert, also kein Hinweis auf Leakage. Vorsicht beim Lesen: Der Random Forest überschätzt Merkmale mit vielen verschiedenen Werten wie Fare und Alter. Das eigentliche Warnsignal steht in der Ausgabe oben: Train-Accuracy 0,985 gegen CV 0,80 ist Overfitting, das Schritt 10 mit grösseren Blättern bremst.

### Schritt 10 · Regularisierung und Hyperparameter-Suche

Gegen das Overfitting helfen flachere Bäume (`max_depth`) und grössere Blätter (`min_samples_leaf`: Ein Blatt muss mindestens so viele Passagiere enthalten). Das Grid hat 4 × 3 = 12 Kombinationen, mit 5 Folds sind das 60 Trainingsläufe. Bewertet wird nach AUC.

In [ ]:
# 10 · Regularisierung und Hyperparameter-Suche: flachere Bäume, grössere Blätter
grid = {"clf__max_depth": [4, 6, 8, None], "clf__min_samples_leaf": [1, 3, 5]}
search = GridSearchCV(model, grid, cv=cv, scoring="roc_auc").fit(X_tr, y_tr)
print(search.best_params_, "| CV-AUC:", round(search.best_score_, 3))

Gewonnen hat «Bäume ohne Tiefenlimit, aber mindestens 3 Passagiere pro Blatt». Die CV-AUC steigt von 0,866 auf 0,88.

### Schritt 11 · Bestes Modell genau einmal auf dem Testset prüfen

`predict_proba` liefert die Überlebenswahrscheinlichkeit, die braucht man für AUC und ROC-Kurve. `classification_report` zeigt Precision, Recall und F1 für beide Klassen.

In [ ]:
# 11 · Bestes Modell genau einmal auf dem Testset prüfen
proba = search.predict_proba(X_te)[:, 1]
print(classification_report(y_te, search.predict(X_te), target_names=["gestorben", "überlebt"], digits=2))
print("Test-AUC:", round(roc_auc_score(y_te, proba), 3))
RocCurveDisplay.from_predictions(y_te, proba)
plt.show()

*Grafik:* ROC-Kurve auf den 179 Testpassagieren. Je näher die Kurve an der linken oberen Ecke, desto besser; die Diagonale wäre Zufall.

Die Test-AUC (0,844) liegt etwas unter der CV-AUC (0,88). Die Test-Accuracy (0,80) liegt auf dem CV-Wert des ungetunten Waldes (0,80); das getunte Modell erreichte in der CV rund 0,82 Accuracy, der Test liegt also leicht darunter. Der Recall für «überlebt» ist 0,72: Gut ein Viertel der Überlebenden sagt das Modell falsch als gestorben voraus. Im Vergleich zu Teil 1 (Test-Accuracy 0,79) bringt der aufwendigere Random Forest hier kaum mehr Accuracy. Bei so wenigen Merkmalen ist das einfache Modell eine gute Wahl.

`stratify=y` sorgt dafür, dass Train und Test das gleiche Klassenverhältnis haben. Die Pipeline lernt Imputer und Scaler in jedem Fold nur auf dem Trainingsteil und verhindert so Leakage. Die Vorlage des Dozenten liegt im Kursordner unter `02 - SUPERVISED-ML/code/CAS_ML_Supervised_Learning_2.ipynb`.